# 03 — Data Cleaning & Preprocessing
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:** `data/raw/books.csv`  
**Output:** `data/processed/books_clean.csv`

Pipeline:
1. Load raw data
2. Audit missing values
3. Drop / fill theo từng cột (theo PRD §5)
4. Remove duplicates
5. Outlier filtering
6. Feature engineering
7. Final validation & save

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_PATH  = Path('data/raw/books.csv')
OUT_PATH  = Path('data/processed/books_clean.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Paths OK')

## 1. Load Raw Data

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Raw shape : {df_raw.shape}')
print(f'Columns   : {df_raw.columns.tolist()}')
df_raw.head(3)

## 2. Missing Value Audit

In [ ]:
missing = pd.DataFrame({
    'missing_count': df_raw.isnull().sum(),
    'missing_pct'  : (df_raw.isnull().mean() * 100).round(2),
    'dtype'        : df_raw.dtypes,
})
print('=== Missing Value Report (raw) ===')
missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)

## 3. Cleaning Pipeline

Thực hiện theo bảng trong PRD §5 Step 2.

In [ ]:
df = df_raw.copy()
log = []  # track mỗi bước drop bao nhiêu rows

def log_step(name, before, after, action=''):
    removed = before - after
    log.append({'step': name, 'before': before, 'after': after,
                'removed': removed, 'action': action})
    print(f'[{name}] {before:,} → {after:,}  (removed {removed:,})  {action}')

n0 = len(df)
print(f'Start: {n0:,} rows')
print()

### 3.1 Drop rows with missing `description`

In [ ]:
before = len(df)
df = df.dropna(subset=['description'])

# Lọc mô tả boilerplate public-domain
BOILERPLATE = [
    "this work has been selected by scholars",
    "this is a reproduction of",
    "we appreciate your understanding of the imperfections",
]
_low = df["description"].str.lower()
is_boiler = _low.apply(lambda s: any(p in s for p in BOILERPLATE))
df = df[~is_boiler]

# Also drop description that is too short (< 10 words — likely not useful)
df = df[df['description'].str.split().str.len() >= 10]

log_step('Drop missing/short/boilerplate description', before, len(df),
         'drop: NaN, boilerplate, or < 10 words')

### 3.2 Drop duplicate `isbn13`

In [ ]:
before = len(df)

# Drop rows where isbn13 is missing first
df = df.dropna(subset=['isbn13'])

# Keep first occurrence of isbn13
df = df.drop_duplicates(subset=['isbn13'], keep='first')

# Khử trùng lặp edition (tác phẩm) theo title + author
_dlen = df["description"].str.split().str.len()
_key = df["title"].str.lower().str.strip() + "::" + df["authors"].str.lower().str.strip()
df = (
    df.assign(_dlen=_dlen, _key=_key)
      .sort_values("_dlen", ascending=False)
      .drop_duplicates("_key", keep="first")
      .drop(columns=["_dlen", "_key"])
      .sort_index()
)

log_step('Deduplicate isbn13 & edition', before, len(df), 'keep first isbn13 + longest description per title-author')

### 3.3 Outlier filter: `num_pages`

In [ ]:
before = len(df)

# Đổi 0 thành NaN (Google Books dùng 0 làm ký hiệu "không rõ số trang")
df['num_pages'] = df['num_pages'].replace(0, pd.NA)

# Keep only rows where num_pages is in [10, 1500] or missing
mask_pages = (
    df['num_pages'].isna() |
    ((df['num_pages'] >= 10) & (df['num_pages'] <= 1500))
)
df = df[mask_pages]

log_step('Filter num_pages outliers (0->NaN)', before, len(df), 'keep 10–1500 or NaN')

### 3.4 Outlier filter: `published_year`

In [ ]:
before = len(df)

mask_year = (
    df['published_year'].isna() |
    ((df['published_year'] >= 1800) & (df['published_year'] <= 2026))
)
df = df[mask_year]

log_step('Filter published_year outliers', before, len(df), 'keep 1800–2026 or NaN')

### 3.5 Fill missing values

In [ ]:
PLACEHOLDER_THUMB = 'https://via.placeholder.com/128x192.png?text=No+Cover'

# thumbnail → placeholder URL
df['thumbnail'] = df['thumbnail'].fillna(PLACEHOLDER_THUMB)

# average_rating → median of remaining values
rating_median = df['average_rating'].median()
df['average_rating'] = df['average_rating'].fillna(rating_median)

# categories → 'Unknown'
df['categories'] = df['categories'].fillna('Unknown')

# authors → 'Unknown'
df['authors'] = df['authors'].fillna('Unknown')

# num_pages → median
pages_median = df['num_pages'].median()
df['num_pages'] = df['num_pages'].fillna(pages_median)

# published_year → median cast to int
year_median = int(df['published_year'].median())
df['published_year'] = df['published_year'].fillna(year_median).astype(int)

print('Fill summary:')
print(f'  thumbnail fill value   : placeholder URL')
print(f'  average_rating median  : {rating_median:.2f}')
print(f'  num_pages median       : {pages_median:.0f}')
print(f'  published_year median  : {year_median}')
print(f'  categories fill value  : "Unknown"')
print()
print(f'Remaining missing after fills:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 4. Feature Engineering

In [ ]:
# description_length: word count
df['description_length'] = df['description'].str.split().str.len()

# book_age: years since publication (CURRENT_YEAR = 2026)
df['book_age'] = 2026 - df['published_year']

# tag_clean: lowercased, stripped categories
df['tag_clean'] = df['categories'].str.lower().str.strip()

print('New features added:')
print(df[['description_length', 'book_age', 'tag_clean']].describe(include='all'))

## 5. Clean Description Text

Chỉ normalize nhẹ — **không** stemming/lemmatization vì:
- TF-IDF hoạt động tốt trên raw text với `TfidfVectorizer` built-in tokenizer
- BGE embedding model xử lý raw text tốt hơn text đã bị biến đổi

In [ ]:
import re

def clean_text(text: str) -> str:
    """Light normalization: strip HTML tags, collapse whitespace, fix encoding."""
    if not isinstance(text, str):
        return ''
    # Remove HTML tags (e.g. <br>, <b>)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Collapse multiple whitespace / newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['description'] = df['description'].apply(clean_text)

# Re-compute description_length after cleaning
df['description_length'] = df['description'].str.split().str.len()

# Drop rows where description became empty after cleaning
before = len(df)
df = df[df['description_length'] >= 10]
log_step('Drop empty description after clean_text', before, len(df))

print(f'\nSample cleaned description:')
print(df['description'].iloc[0][:300])

## 6. Select & Order Final Columns

In [ ]:
# Required columns theo PRD §5 Step 5
FINAL_COLS = [
    'isbn13',
    'title',
    'authors',
    'description',
    'categories',
    'tag_clean',
    'thumbnail',
    'average_rating',
    'ratings_count',
    'published_year',
    'num_pages',
    'description_length',
    'book_age',
]

# Only keep columns that actually exist in df
final_cols = [c for c in FINAL_COLS if c in df.columns]
missing_cols = [c for c in FINAL_COLS if c not in df.columns]
if missing_cols:
    print(f'Warning — columns not found in raw data: {missing_cols}')
    print('  → skipping those columns (check raw CSV header names)')

df_final = df[final_cols].reset_index(drop=True)
print(f'Final columns  : {df_final.columns.tolist()}')
print(f'Final shape    : {df_final.shape}')

## 7. Final Validation

In [ ]:
print('=== Final Validation ===')

# 1. No missing in mandatory columns
mandatory = ['isbn13', 'title', 'description', 'categories', 'thumbnail', 'average_rating']
mandatory_present = [c for c in mandatory if c in df_final.columns]
missing_mandatory = df_final[mandatory_present].isnull().sum()
print('Missing in mandatory columns:')
print(missing_mandatory)
assert missing_mandatory.sum() == 0, 'ERROR: mandatory columns still have nulls!'

# 2. isbn13 unique
assert df_final['isbn13'].nunique() == len(df_final), 'ERROR: duplicate isbn13!'
print('✓ isbn13 all unique')

# 3. description_length >= 10
assert (df_final['description_length'] >= 10).all(), 'ERROR: short descriptions found!'
print('✓ All descriptions >= 10 words')

# 4. average_rating in [0, 5]
assert df_final['average_rating'].between(0, 5).all(), 'ERROR: rating out of range!'
print('✓ average_rating in [0, 5]')

print()
print('=== Pipeline Summary ===')
pd.DataFrame(log)

## 8. Save Clean Dataset

In [ ]:
df_final.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Rows   : {len(df_final):,}')
print(f'Columns: {len(df_final.columns)}')
print()
df_final.head(3)

## 9. Quick Stats on Clean Dataset

In [ ]:
print('=== Numeric Summary ===')
display_cols = ['average_rating', 'description_length', 'book_age', 'num_pages']
display_cols_present = [c for c in display_cols if c in df_final.columns]
df_final[display_cols_present].describe().round(2)

In [ ]:
print('=== Category Distribution (Top 10) ===')
print(df_final['categories'].value_counts().head(10).to_string())

---
## Done ✓

**Output:** `data/processed/books_clean.csv`  
**Next step:** `04_retrieval_baseline.ipynb` — TF-IDF + BM25 trên dataset này.